In [48]:
import os, sys

PSSPY_location = r'C:\Program Files\PTI\PSSE36\36.5\PSSPY314'
PSSE_location = r'C:\Program Files\PTI\PSSE36\36.5\PSSBIN'
sys.path.append(PSSPY_location)
os.environ['PATH'] += ';' + PSSPY_location
os.environ['PATH'] += ';' + PSSE_location

import psse36
psse36.set_minor(5)
import psspy
from psspy import _i, _f, _s
import redirect
import numpy as np
import pandas as pd
import re
import glob

#Start program
_i=psspy.getdefaultint()
_f=psspy.getdefaultreal()
_s=psspy.getdefaultchar()
redirect.psse2py()
psspy.psseinit(800000)

# suppress output
# psspy.report_output(6,'',[])
# psspy.progress_output(6,'',[])
# psspy.alert_output(6,'',[])
# psspy.prompt_output(6,'',[])

0

In [49]:
case = "Maui24_DM_r8.sav"
input_excel = "Maui_details.xlsx"
psspy.case(case)

#get gen info
gen_bus = psspy.amachint(-1, 4, 'NUMBER')[1][0]
gen_id = psspy.amachchar(-1, 4, 'ID')[1][0]
gen_name = psspy.amachchar(-1, 4, 'NAME')[1][0]
gen_mod = psspy.amachint(-1, 4, 'WMOD')[1][0]
gen_status = psspy.amachint(-1, 4, 'STATUS')[1][0]
gen_Pmax = psspy.amachreal(-1, 4, 'PMAX')[1][0]
gen_list = list(map(list, zip(*[gen_bus, gen_name, gen_id, gen_status, gen_Pmax, gen_mod])))
gen_df = pd.DataFrame(gen_list, columns = ["Bus Number", "Bus Name", "Gen ID", "Status", "PMAX", "WMOD"])
   


 Two-winding transformer ckt "1" from 70105 [ULUPALKU_Z  34.500] to 70106 [ULUPALKU_B  0.5000]:
   name "ISU" is assigned to another transformer; the name of this transformer is changed to "ISU_2"

 Two-winding transformer ckt "1" from 84 [LAHALUNA    69.000] to 234 [LAHAINA 2   12.470]:
   name "LAHAINA 2" is assigned to another transformer; the name of this transformer is changed to "LAHAINA 2_2"

 MECO 2024 DAY MIN - 33.821 MW
 99.74 MW DG PV - TOTAL SYSTEM LOAD IN CASE ~140.96 MW

 The Saved Case in file C:\Users\ckauffma\Documents\Tools\System Strength Assessment\formally organized tools\Netstrength_3\src\system_strength_tool\model_data\Maui24_DM_r8.sav was saved on MON, APR 07 2025   7:37


In [50]:
excel_df = pd.read_excel(input_excel, sheet_name = "Generator")
excel_df = excel_df.rename(columns={"busname": "Bus Number"})
excel_df = excel_df[["genname", "Bus Number", "Gen_Type"]]
excel_df

,genname,Bus Number,Gen_Type
0,Kahului3,103,Gas
1,Kahului4,104,Gas
2,Maalaea01,105,Gas
3,Maalaea02,105,Gas
4,Maalaea03,105,Gas
...,...,...,...
84,DPV57,4250,PV-1
85,DPV58,4430,PV-1
86,DPV59,5340,PV-1
87,DPV60,7290,PV-1


In [53]:
merged_df = gen_df.merge(excel_df, on = "Bus Number", how="outer", suffixes=("_case", "_excel"))
merged_df.to_excel("Maui_merged_case_and_excel.xlsx", index = False)

In [62]:
from psspy import _i, _f, _s

#iterate through all machines and determine update to WMOD
count = 0
for index, gen in merged_df[~merged_df["Gen ID"].isnull()].iterrows():
    if "PV" in str(gen["Gen_Type"]) or "Wind" in str(gen["Gen_Type"]):
        print(gen["Bus Number"])
        print(gen["Gen ID"])
        turn_on = 1
        #change WMOD to 1
        ierr = psspy.machine_chng_5(gen["Bus Number"],gen["Gen ID"], [_i,_i,_i,_i,_i,1,_i], [_f] * 17,[_s,_s])
        if ierr !=0:
            print("IERR: " + str(ierr))
        count += 1

psspy.save(case.replace(".sav", "_WMOD.sav"))
print(count)

13
P1

 Power flow data changed for machine "P1" at bus 13 [KULA 12     12.470]:
 X--ORIGINAL--X  X-NEW VALUE--X  DATA ITEM
          0               1      WMOD
13
P2

 Power flow data changed for machine "P2" at bus 13 [KULA 12     12.470]:
 X--ORIGINAL--X  X-NEW VALUE--X  DATA ITEM
          0               1      WMOD
13
P3

 Power flow data changed for machine "P3" at bus 13 [KULA 12     12.470]:
 X--ORIGINAL--X  X-NEW VALUE--X  DATA ITEM
          0               1      WMOD
98
P1

 Power flow data changed for machine "P1" at bus 98 [KAUHIKOA    12.470]:
 X--ORIGINAL--X  X-NEW VALUE--X  DATA ITEM
          0               1      WMOD
98
P3

 Power flow data changed for machine "P3" at bus 98 [KAUHIKOA    12.470]:
 X--ORIGINAL--X  X-NEW VALUE--X  DATA ITEM
          0               1      WMOD
129
P1

 Power flow data changed for machine "P1" at bus 129 [NAPILA12    12.470]:
 X--ORIGINAL--X  X-NEW VALUE--X  DATA ITEM
          0               1      WMOD
129
P2

 Power flow data c